In [8]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor
from joblib import dump, load

for dirname, _, filenames in os.walk("./"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

./homes_third1.csv
./homes_third2.csv
./final_model.joblib
./homes.csv
./Project_2.ipynb
./homes_third3.csv
./data_scraping.py
./.ipynb_checkpoints/homes5-checkpoint.csv
./.ipynb_checkpoints/homes2-checkpoint.csv
./.ipynb_checkpoints/isntworking-checkpoint.ipynb
./.ipynb_checkpoints/homes-checkpoint.csv
./.ipynb_checkpoints/Project_2_dupe_copy-checkpoint.ipynb
./.ipynb_checkpoints/Project_2-checkpoint.ipynb
./.ipynb_checkpoints/homes_third11-checkpoint.csv
./.ipynb_checkpoints/homes1-checkpoint.csv
./.ipynb_checkpoints/homes_third2-checkpoint.csv
./.ipynb_checkpoints/homes4-checkpoint.csv
./.ipynb_checkpoints/data_scraping-checkpoint.py
./.ipynb_checkpoints/homes_third3-checkpoint.csv
./.ipynb_checkpoints/homes6-checkpoint.csv
./.ipynb_checkpoints/homes_third1-checkpoint.csv
./.ipynb_checkpoints/untitled-checkpoint.py


# 1. Frame the problem
Using the customer description, Define the problem your trying to solve in your own words (remember this is not technial but must be specific so the customer understands the project

We wish to use past housing data to predict the price of a home based on its attributes, limiting the analysis to a particular city. The city we are using for this is Pheonix, Arizona. We will use attributes from houses listed or sold up to at most two years ago, such as bedrooms, bathrooms, and square footage, as well as their prices, to construct a model that takes in those attributes and returns a predicted price.

# 2. Get the Data 
Define how you recieved the data (provided, gathered..)

In [9]:
'''
The following code was used to divide the data into 3 separate files so that it could be pushed to GitHub:

df = pd.read_csv('homes.csv', dtype=str)
zero_rows = df.iloc[::3]
one_rows = df.iloc[1::3]
two_rows = df.iloc[2::3]
zero_rows.to_csv('homes_third1.csv', index=False)
one_rows.to_csv('homes_third2.csv', index=False)
two_rows.to_csv('homes_third3.csv', index=False)
    # For now, 'homes.csv' is too large to upload to git, so it is split into
    # homes_third1.csv, homes_third2.csv, and homes_third3.csv so it can be pushed
'''

"\nThe following code was used to divide the data into 3 separate files so that it could be pushed to GitHub:\n\ndf = pd.read_csv('homes.csv', dtype=str)\nzero_rows = df.iloc[::3]\none_rows = df.iloc[1::3]\ntwo_rows = df.iloc[2::3]\nzero_rows.to_csv('homes_third1.csv', index=False)\none_rows.to_csv('homes_third2.csv', index=False)\ntwo_rows.to_csv('homes_third3.csv', index=False)\n    # For now, 'homes.csv' is too large to upload to git, so it is split into\n    # homes_third1.csv, homes_third2.csv, and homes_third3.csv so it can be pushed\n"

In [10]:
'''
The following code can be used to reconstruct 'homes.csv' from 'homes_third1.csv', 'homes_third2.csv', and 'homes_third3.csv':

df1 = pd.read_csv('homes_third1.csv', dtype=str)
df2 = pd.read_csv('homes_third2.csv', dtype=str)
df3 = pd.read_csv('homes_third3.csv', dtype=str)
df = pd.concat([df1, df2], ignore_index=True)
df = pd.concat([df, df3], ignore_index=True)
df.iloc[::3] = df1
df.iloc[1::3] = df2
df.iloc[2::3] = df3
df.to_csv('homes.csv', index=False)
'''

"\nThe following code can be used to reconstruct 'homes.csv' from 'homes_third1.csv', 'homes_third2.csv', and 'homes_third3.csv':\n\ndf1 = pd.read_csv('homes_third1.csv', dtype=str)\ndf2 = pd.read_csv('homes_third2.csv', dtype=str)\ndf3 = pd.read_csv('homes_third3.csv', dtype=str)\ndf = pd.concat([df1, df2], ignore_index=True)\ndf = pd.concat([df, df3], ignore_index=True)\ndf.iloc[::3] = df1\ndf.iloc[1::3] = df2\ndf.iloc[2::3] = df3\ndf.to_csv('homes.csv', index=False)\n"

We used the HomeHarvest package to scrape housing data from realtor.com. Our parameters searched for houses sold and listed in the past 2 years, looping over every zip code in Pheonix; the code used to do this is located in data_scraping.py. Before any preprocessing, this means that some entries also include houses not in Pheonix that will likely need to be removed.

# 3. Explore the Data
Gain insights into the data you have from step 2, making sure to identify any bias

In [11]:
df_explore = pd.read_csv('homes.csv')
df_explore = df_explore[df_explore['status'] == 'SOLD']

df_explore['sold_price'] = pd.to_numeric(df_explore['sold_price'], errors='coerce')
df_explore['sqft'] = pd.to_numeric(df_explore['sqft'], errors='coerce')
df_explore['year_built'] = pd.to_numeric(df_explore['year_built'], errors='coerce')
df_explore['beds'] = pd.to_numeric(df_explore['beds'], errors='coerce')

df_explore.dropna(subset=['sold_price'], inplace=True)
df_explore.dropna(subset=['sqft'], inplace=True)
df_explore.dropna(subset=['year_built'], inplace=True)
df_explore.dropna(subset=['beds'], inplace=True)

X_year = df_explore[['year_built']]
X_sqft = df_explore[['sqft']]
X_beds = df_explore[['beds']]
Y = df_explore['sold_price']

model_year = LinearRegression()
model_year.fit(X_year, Y)
model_sqft = LinearRegression()
model_sqft.fit(X_sqft, Y)
model_beds = LinearRegression()
model_beds.fit(X_beds, Y)

print('Attribute', 'Slope', 'Intercept', 'R squared')
print('Year', model_year.coef_[0], model_year.intercept_, model_year.score(X_year, Y))
print('Sqft', model_sqft.coef_[0], model_sqft.intercept_, model_sqft.score(X_sqft, Y))
print('Beds', model_beds.coef_[0], model_beds.intercept_, model_beds.score(X_beds, Y))

/tmp/ipykernel_3518821/2950656462.py:1: DtypeWarning: Columns (5,55) have mixed types. Specify dtype option on import or set low_memory=False.
  df_explore = pd.read_csv('homes.csv')


Attribute Slope Intercept R squared
Year 3383.292404166018 -6148245.921673668 0.02510728031112608
Sqft 374.8354103231598 -119325.64802457788 0.5066223652122979
Beds 215508.2931265003 -113583.0735900132 0.17671220844156732


We first note that we expect square footage, number of bedrooms, and number of bathrooms to all have a strong positive correlation with price, and we are biased towards considering these attributes in particular. To get a first look at the data, we picked a few attributes and performed a linear regression using that attribute against sell price for sold houses. We picked two of the attributes we suspected would have a strong positive correlation, square footage and bedrooms, as well as the year sold, for which we were unsure if there would be a strong correlation in either direction. The r^2 value for predicting sell price based off of year was 0.025, which is fairly weak, but strong enough that we will likely consider using it for our model. The r^2 values for square footage and bedrooms are 0.507 and 0.177 respectively, both of which indicate correlations of notable significance. The slopes of both of those regressions were also positive, as predicted.

We should also note that these are all crude estimates, seeing as we only used sold houses, dropped NaNs from all attributes concurrently instead of individually, and have not performed the necessary preprocessing to restrict to houses in Pheonix.

# 4.Prepare the Data


Apply any data transformations and explain what and why


In [12]:
df = pd.read_csv('homes.csv', dtype=str)
df = df[df['city'] == 'Phoenix']
    # The data includes some houses not in Pheonix; we wish to ignore those
df = df[df['status'] == 'SOLD']
    # To make it simpler, we will only look at houses that have been sold
df.dropna(subset=['sold_price'], inplace=True) 
df = df[['sold_price', 'style', 'beds', 'full_baths', 'sqft', 'year_built', 'stories', 'parking_garage']]
    # We include all data that we think will be both useful and usable
    # For example, property_id is probably not useful,
    # but text might be useful but is also hard to use
df = pd.get_dummies(df, columns=['style'])
    # We perform one hot encoding for style of home to use the categorical data
df = df.apply(pd.to_numeric, errors='coerce')
    # Convert all data from strings to floats
df.dropna(inplace=True)
print(df.shape[0])
print(df.head(20))

14031
    sold_price  beds  full_baths    sqft  year_built  stories  parking_garage  \
0       515000   4.0         2.0  1957.0      1980.0      1.0             2.0   
2       850000   3.0         2.0  1654.0      1935.0      1.0             2.0   
4       623000   2.0         1.0  1076.0      1937.0      1.0             1.0   
9       808000   3.0         2.0  1495.0      1940.0      1.0             1.0   
13      830000   3.0         2.0  2454.0      2004.0      2.0             2.0   
14      590000   3.0         2.0  1361.0      1947.0      1.0             2.0   
16     1000000   3.0         2.0  2073.0      1948.0      1.0             3.0   
17      885000   2.0         2.0  1498.0      2016.0     15.0             2.0   
18     1650000   4.0         3.0  3020.0      1930.0      1.0             2.0   
22      915000   3.0         2.0  1889.0      1936.0      1.0             2.0   
26      870000   3.0         2.0  1656.0      1935.0      1.0             1.0   
27      800000   4.0  

In [13]:
X = df[df.columns[1:]]
y = df['sold_price']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
    # Define the training and testing datasets

# 5. Model the data
Using selected ML models, experment with your choices and describe your findings. Finish by selecting a Model to continue with


In [90]:
    # Model using KNN
scaler = StandardScaler()
X_train_normalized = scaler.fit_transform(X_train)
X_test_normalized = scaler.transform(X_test)
model_knn = KNeighborsRegressor(n_neighbors=10)
model_knn.fit(X_train_normalized, y_train)
y_pred = model_knn.predict(X_test_normalized)
r2 = r2_score(y_test, y_pred)
print(r2)

0.7208642593914419


In [101]:
    # Model using random forests
model_random_forest = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    random_state=42
)
model_random_forest.fit(X_train, y_train)
y_pred = model_random_forest.predict(X_test)
r2 = r2_score(y_test, y_pred)
print(r2)

0.7680268955419854
0.7680268955419854


# 6. Fine Tune the Model

With the select model descibe the steps taken to acheve the best rusults possiable 


The random forest yields a better r^2 value before fine tuning, so we will continue using it.

In [98]:
model_random_forest = RandomForestRegressor(random_state=42)
parameter_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 25],
    'min_samples_split': [2, 6, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}
grid_search = GridSearchCV(
    estimator=model_random_forest,
    param_grid=parameter_grid,
    scoring='r2',
)
    # Search through different comvinations of parameters
grid_search.fit(X_train, y_train)
best_random_forest = grid_search.best_estimator_
y_pred = best_random_forest.predict(X_test)
dump(best_random_forest, 'final_model.joblib')
r2 = r2_score(y_test, y_pred)
print(r2)

0.7628608284173518


The r^2 value of 0.763 after parameter tuning is worse than the original value of 0.768, so we will save the original model to memory.

In [102]:
model_random_forest = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    random_state=42
)
model_random_forest.fit(X_train, y_train)
y_pred = model_random_forest.predict(X_test)
dump(model_random_forest, 'final_model.joblib')
    # Save the original model to memory

['final_model.joblib']

# 7. Present
In a customer faceing Document provide summery of finding and detail approach taken


The given problem was to develop a model that given attributes of a house in Phoenix could predict its selling price. We began by exploring patterns in the data, estimating the strength of linear relationships of between a feature and selling price over 3 features that we were interested in using for our model. We identified strong relationships from this.

When selecting features to use, we picked features that we predicted would both be helpful and usable. For example, text descriptions were dropped because they would be hard to use, while property IDs were dropped because they were irrelevant. Due to an abundance of data, we dealt with missing values by just dropping every row with a missing value. We trained two models after splitting the data: a K-nearest neighbors model and a random forest. After comparing their accuracy, we moved forward with the random forest. Finally, after tuning parameters to improve our model, we did not get better results than the original model, and so we kept the original model. This yielded a random forest with r^2 value 76.8% on the test data for predicting selling price.

# 8. Launch the Model System
Define your production run code, This should be self susficent and require only your model pramaters 


In [2]:
def infrence(beds: int, full_bathrooms: int, sqft: float, year_built: int, stories: int, parking_garages: int, style: str) -> float:
    styles = ['APARTMENT', 'COMMERCIAL', 'CONDO', 'CONDOS', 'COOP', 'LAND', 'MOBILE', 'MULTI_FAMILY', 'OTHER', 'SINGLE_FAMILY', 'TOWNHOMES']
        # style must be a member of the array styles
    if style not in styles:
        style = 'OTHER'
    import os
    import pandas as pd
    from sklearn.ensemble import RandomForestRegressor
    from joblib import load
    import numpy as np
    import warnings
    warnings.filterwarnings("ignore", category=UserWarning)
    model = load('final_model.joblib')
    input_attributes = [[float(beds), float(full_bathrooms), sqft, float(year_built), float(stories), float(parking_garages)]
        + [float(style == s) for s in styles]]
    prediction = model.predict(input_attributes)
    return prediction[0]

In [3]:
print(infrence(1, 2, 2000, 2000, 1, 1, 'SINGLE_FAMILY'))

660333.3333333334
